In [36]:
import pandas as pd
import random
from collections import deque
import copy
import os

In [37]:
df = pd.read_csv('dataset.csv')
severity_df = pd.read_csv('Symptom-severity (1).csv')
precaution_df = pd.read_csv('symptom_precaution.csv')
print("Files loaded")
print(df.columns)

Files loaded
Index(['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4',
       'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9',
       'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14',
       'Symptom_15', 'Symptom_16', 'Symptom_17'],
      dtype='object')


In [6]:
df = pd.read_csv('/content/dataset.csv')
severity_df = pd.read_csv('/content/Symptom-severity (1).csv')
precaution_df = pd.read_csv('/content/symptom_precaution.csv')

print("Datasets Loaded Successfully")

Datasets Loaded Successfully


In [39]:
# Check columns first (debug)
print(df.columns)

# Use correct column name
fungal_df = df[df['Disease'].str.lower() == 'fungal infection']

# Extract symptom columns
symptom_cols = [c for c in df.columns if 'symptom' in c.lower()]

fungal_symptoms = set()

for col in symptom_cols:
    # Clean symptoms by stripping whitespace
    fungal_symptoms.update(fungal_df[col].dropna().apply(lambda x: x.strip()).values)

fungal_symptoms = list(fungal_symptoms)

print("Total symptoms:", len(fungal_symptoms))

Index(['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4',
       'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9',
       'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14',
       'Symptom_15', 'Symptom_16', 'Symptom_17'],
      dtype='object')
Total symptoms: 4


In [40]:
severity_df.columns = ['Symptom', 'Weight']
severity = dict(zip(severity_df['Symptom'], severity_df['Weight']))

# keep only fungal symptoms
severity = {k: severity[k] for k in fungal_symptoms if k in severity}

print("Filtered symptoms:", len(severity))

Filtered symptoms: 3


In [41]:
precaution_df.columns = ["Disease","P1","P2","P3","P4"]
precaution_df["Disease"] = precaution_df["Disease"].str.strip()

fungal_precautions = precaution_df[
    precaution_df["Disease"].str.lower() == "fungal infection"
].iloc[0,1:].dropna().tolist()

In [42]:
variables = ["S1","S2","S3","S4"]
domains = {
    v: [s for s in severity if severity[s] >= 2]
    for v in variables
}
constraints = [(Xi,Xj) for Xi in variables for Xj in variables if Xi!=Xj]

In [43]:
# C1 (Binary): No duplicate symptoms (Xi ≠ Xj)
# C2 (Binary): If itching appears, skin_rash must be allowed in neighbors
# C3 (Unary): Remove very low severity (<2)
# C4 (Global): At least one severity ≥ 3
# C5 (Global): Total severity ≤ 20


def is_consistent(assignment):
    vals = list(assignment.values())

    # C1: Unique
    if len(vals) != len(set(vals)):
        return False

    # C2: itching → skin_rash
    if "itching" in vals and "skin_rash" not in vals:
        return False

    # Global constraints only when full
    if len(assignment) == 4:

        # C4: at least one severity >=3
        if not any(severity[s] >= 3 for s in vals):
            return False

        # C5: total ≤ 20
        if sum(severity[s] for s in vals) > 20:
            return False

    return True

In [48]:
def revise(csp, Xi, Xj):
    revised = False

    for x in csp[Xi][:]:
        if all(x == y for y in csp[Xj]):
            csp[Xi].remove(x)
            revised = True
    return revised


def ac3(csp, constraints):
    queue = deque(constraints)
    while queue:
        Xi, Xj = queue.popleft()
        if revise(csp, Xi, Xj):
            if len(csp[Xi]) == 0:
                return False
            for Xk,_ in constraints:
                if Xk != Xi:
                    queue.append((Xk,Xi))
    return True
csp = copy.deepcopy(domains)

print("Before AC-3:", {k: len(v) for k,v in csp.items()})
ac3(csp, constraints)
print("After AC-3:", {k: len(v) for k,v in csp.items()})

Before AC-3: {'S1': 2, 'S2': 2, 'S3': 2, 'S4': 2}
After AC-3: {'S1': 2, 'S2': 2, 'S3': 2, 'S4': 2}


In [51]:
bcount = 0
def backtrack(assignment):
    global bcount
    if len(assignment) == 4:
        return assignment
    var = [v for v in variables if v not in assignment][0]

    for val in csp[var]:
        new = assignment.copy()
        new[var] = val
        if is_consistent(new):
            res = backtrack(new)
            if res:
                return res

        bcount += 1
    return None


solution = backtrack({})
print("BT:", solution)
print("Backtracks:", bcount)

BT: None
Backtracks: 10


In [64]:
fcount = 0

def fcheck(assignment):
    global fcount
    if len(assignment) == 4:
        return assignment
    var = [v for v in variables if v not in assignment][0]

    for val in csp[var]:
        new = assignment.copy()
        new[var] = val

        if is_consistent(new):
            fail = False
            for future in variables:
                if future not in new:
                    if not any(
                        is_consistent({**new, future: v})
                        for v in csp[future]
                    ):
                        fail = True
                        break

            if not fail:
                res = fcheck(new)
                if res:
                    return res

        fcount += 1
    return None

solution = fcheck({})
print("FC:", solution)
print("Backtracks:", fcount)

FC: None
Backtracks: 6


In [65]:
mcount = 0

def select_mrv(assignment):
    unassigned = [v for v in variables if v not in assignment]
    return min(unassigned, key=lambda x: len(csp[x]))

def backtrack_mrv(assignment):
    global mcount
    if len(assignment) == 4:
        return assignment
    var = select_mrv(assignment)

    for val in csp[var]:
        new = assignment.copy()
        new[var] = val

        if is_consistent(new):
            res = backtrack_mrv(new)
            if res:
                return res
        mcount += 1
    return None


solution = backtrack_mrv({})
print("MRV:", solution)
print("Backtracks:", mcount)

MRV: None
Backtracks: 10


In [71]:
def count_conflicts(a):
    return 0 if is_consistent(a) else 1

def min_conflicts(steps=100):
    a = {v: random.choice(csp[v]) for v in variables}

    for _ in range(steps):
        if count_conflicts(a) == 0:
            return a
        var = random.choice(variables)
        a[var] = min(csp[var], key=lambda val: count_conflicts({**a,var:val}))

    return a
sol_mc = min_conflicts()
print("Min-Conflicts:", sol_mc)

Min-Conflicts: {'S1': 'nodal_skin_eruptions', 'S2': 'nodal_skin_eruptions', 'S3': 'nodal_skin_eruptions', 'S4': 'nodal_skin_eruptions'}


In [69]:
print("\nFINAL SOLUTION (MRV BEST):")
print(sol_mrv)

print("\nPrecautions:")
for p in fungal_precautions:
    print("- ", p)

print("\nBacktracking Comparison:")
print("BT:", bt_count)
print("FC:", fc_count)
print("MRV:", mrv_count)

print("\nFinal Violations:", violations[-1])


FINAL SOLUTION (MRV BEST):
None

Precautions:
-  bath twice
-  use detol or neem in bathing water
-  keep infected area dry
-  use clean cloths

Backtracking Comparison:
BT: 10
FC: 6
MRV: 10

Final Violations: 1


# New Section